In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

c:\Users\Roni\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MAX_ITERATIONS = 10
MODEL = "gemini-2.5-flash"

In [ ]:
#Tools (Lanchain @tool decorator)
@tool
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    product = product.lower()
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)

@tool
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price.
    Available tiers: bronze, silver, gold."""
    discount_tier = discount_tier.lower()
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)

In [4]:
# --- Agent Loop ---


from langsmith import traceable

#from asyncio import tools


@traceable(name="LangChain Agent Loop")
def run_agent(question: str):
    tools = [get_product_price, apply_discount]
    tools_dict = {t.name: t for t in tools}

    
    llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
    )

    llm_with_tools = llm.bind_tools(tools)
    print(type(llm))

    print(f"Question: {question}")
    print("=" * 60)

    messages = [
        SystemMessage(
            content=(
                "You are a helpful shopping assistant. "
                "You have access to a product catalog tool "
                "and a discount tool.\n\n"
                "STRICT RULES — you must follow these exactly:\n"
                "1. NEVER guess or assume any product price. "
                "You MUST call get_product_price first to get the real price.\n"
                "2. Only call apply_discount AFTER you have received "
                "a price from get_product_price. Pass the exact price "
                "returned by get_product_price — do NOT pass a made-up number.\n"
                "3. NEVER calculate discounts yourself using math. "
                "Always use the apply_discount tool.\n"
                "4. If the user does not specify a discount tier, "
                "ask them which tier to use — do NOT assume one."
            )
        ),
        HumanMessage(content=question),
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")

        ai_message = llm_with_tools.invoke(messages)

        tool_calls = ai_message.tool_calls

        # If no tool calls, this is the final answer
        if not tool_calls:
            print(f"\nFinal Answer: {ai_message.content}")
            return ai_message.content

        # Process only the FIRST tool call — force one tool per iteration
        tool_call = tool_calls[0]
        tool_name = tool_call.get("name")
        tool_args = tool_call.get("args", {})
        tool_call_id = tool_call.get("id")

        print(f"  [Tool Selected] {tool_name} with args: {tool_args}")

        tool_to_use = tools_dict.get(tool_name)
        if tool_to_use is None:
            raise ValueError(f"Tool '{tool_name}' not found")

        observation = tool_to_use.invoke(tool_args)

        print(f"  [Tool Result] {observation}")

        messages.append(ai_message)
        messages.append(
            ToolMessage(content=str(observation), tool_call_id=tool_call_id)
        )

    print("ERROR: Max iterations reached without a final answer")
    return None


In [8]:
if __name__ == "__main__":
    print("Hello LangChain Agent (.bind_tools)!")
    result = run_agent("What is the price of a Laptop with a bronze discount?")

Hello LangChain Agent (.bind_tools)!
<class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>
Question: What is the price of a Laptop with a bronze discount?

--- Iteration 1 ---
  [Tool Selected] get_product_price with args: {'product': 'Laptop'}
    >> Executing get_product_price(product='laptop')
  [Tool Result] 1299.99

--- Iteration 2 ---
  [Tool Selected] apply_discount with args: {'price': 1299.99, 'discount_tier': 'bronze'}
    >> Executing apply_discount(price=1299.99, discount_tier='bronze')
  [Tool Result] 1234.99

--- Iteration 3 ---

Final Answer: [{'type': 'text', 'text': 'The price of a Laptop with a bronze discount is 1234.99.', 'extras': {'signature': 'CoMCAQw51seUIQesMfHT8zEc+gXshI0gWgkA4Azz6ALDsjN2p/n/uQAZXLvUwqwqJ5kUjIMrhsFsaupo0HwaQRd3xZN4dPXiHPppwnEprgNYysGALl8GF4M6niKLQHsOIpztiQxtMqHj7NKmGmIPIlXf7uR/o2p4ZTvdCHHfeUOdloXXS8jpaBf9YGNTYETtzk6aQvMOAQ6jbJ1Z6+qN61v2TArdR2QmKIPjyoRW/MGJtb2EMhL1/BPMPlR/Kektah3E9yZ7bO96m94yJM7u8b7fFCovK19brX1PE23wdB6FTl9aavgdZU6